# K4TQ-F3 "Freddy 3": as fast as a 15-inch FPV quad with its payload

Freddy's airframe (1.2 m, front NACA 4415 / rear NACA 2415) with the Jetson Orin NX and the perception (as A2), 1 kg payload; the drive swept for the top speed — **T-Motor MN505-S KV380 + Aeronaut CAM Carbon 16x13 folding, on the same 6S3P**.

**The target**: the 15-inch FPV quad's top speed carrying a payload (`k4tq_quads.fpv15`, the same catalogue and drive
model: 15x8 on MN5006 KV450, 6S2P Li-ion) — ≈34.5 m/s (124 km/h); K4TQ-F3 must match or exceed it with four motors
running, with a 5 % margin (the model's 1 m/s grid and its uncertainty).

**Off the shelf**: only parts sold today — T-Motor Navigator motors within their rated voltage (MN505-S is 6S only;
MN601-S / MN605-S / MN801-S 6–12S), Aeronaut CAM Carbon folding blades (16x13, 17x11, 17x13, 18x13, 18x9 on the 42 mm
hub) and T-Motor's folding VTOL props (MF2009 20.4x7.3, FA22.2x7.2), packs of Molicel P45B cells.

**The rules** (`k4tq_fast.sweep` / `pick`): a motor bigger than Freddy autonomous (F2)'s; the still-air range at least 90 % of
Freddy autonomous (F2)'s ("more or less the same"; a bigger pack allowed); T/W ≥ 1.6 on the sagged pack, most loaded motor
≤ 75 % in the hover; R3 closes (10 km radius, 2 min hover, 20 % reserve, 8 m/s wind) with ≥ 20 Wh to spare; the root's
actuator (sized for the top speed + 10 %), bearings and locking pin within their ratings; the lightest that passes (then the most R3 margin).

Why the drive and not the airframe: Freddy autonomous's top speed is its propeller's pitch speed, not its power —
the drag at 35 m/s is only ≈0.5 kW of thrust power. A higher pitch (or a higher pack voltage) moves it.
Labels: ESTIMATE / CALC (MOCK BEMT under `DEBUG = True`); the new parts are "to verify".

In [ ]:
DEBUG = True
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)

In [ ]:
import pandas as pd
from IPython.display import Image, display
import k4tq, k4tq_sizing as sz, k4tq_fast as kf, k4tq_quads as kq
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 50)
LETTER, PARENT = "F3", "F2"
print(k4tq.params(LETTER)["title"])
print("target: the 15-inch FPV quad's top speed with payload", kf.target_speed(), "m/s")

## 1. The sweep (motor × propeller × pack) — from `data/k4tq/sweep_F3.csv`; re-run below (10–30 min)

In [ ]:
sw = pd.read_csv(sz.DATA / "sweep_F3.csv")
sw.attrs["target speed [m/s]"] = kf.target_speed()
# sw = kf.sweep(LETTER); sw.to_csv(sz.DATA / "sweep_F3.csv", index=False)       # the full sweep again
cols = ["motor", "prop", "pack", "top speed [m/s]", "take-off mass [kg]", "payload [kg]", "T/W sagged", "R3 margin [Wh]", "still-air range [km]",
        "hover [W]", "stall [m/s]", "actuator reserve", "meets the target", "note"]
sw[[c for c in cols if c in sw]].sort_values("top speed [m/s]", ascending=False).round(2)

In [ ]:
print("the pick:"); kf.pick(LETTER, sw)

## 2. Against Freddy autonomous (F2) and the 15-inch FPV quad

In [ ]:
df = kf.comparison(designs=(PARENT, LETTER))
df.T

## 3. Power and range against speed

In [ ]:
Image(str(kf.figure(designs=(PARENT, LETTER))))

## 4. The parts: what to buy

In [ ]:
p = k4tq.params(LETTER)
cat = k4tq.CATALOGUE
pd.DataFrame({"motor ×4": cat["motor"][p["motor"]], "propeller ×4 (+2 spares)": cat["propeller"][p["prop"]], "ESC ×4": cat[p["esc"]],
              "pack": cat["battery"][p["battery"]]}).T[["mass_kg", "price_eur", "source"]]

## 5. The root and the hover

In [ ]:
v = sz.vehicle(LETTER)
r = sz.root_loads(v)
display(pd.Series({k: r[k] for k in ("case governing the bending", "root moment ultimate [N·m]", "actuator design torque [N·m]", "actuator available at the spar [N·m]",
                    "actuator reserve", "bearing static reserve", "locking pin shear stress, double shear [MPa]", "spar chosen")}))
pd.Series(sz.hover(v))

## Reading it

- **Is it possible?** Yes. ≈45 m/s (162 km/h) against the FPV quad's 124 km/h with 1 kg — 30 % faster — with the
  1 kg payload, the Jetson, and **the range kept**: 59 km still-air against Freddy autonomous's 54 km (R3 radius 12.0
  against 11.9 km, the margin 23 Wh against 24 Wh). No bigger pack needed.
- **What it took** (all on the shelf): the MN505-S KV380 (225 g, 50 A, 6S — +55 g per motor) and the Aeronaut CAM
  Carbon 16x13 folding blades on the 42 mm hub, on the same 6S3P: +0.27 kg over F2. The higher pitch is what lifts the
  top speed (the pitch speed) *and* the cruise (the 16x10 ran at a poor advance ratio at 23 m/s); it costs ≈0.2 kW in
  the hover (1.39 against 1.18 kW). The hover T/W rises to 1.76 (F2: 1.60, just under the rule).
- **Watch**: the actuator now sized for 50 m/s (top + 10 %): reserve 1.9; the all-moving wing's divergence and flutter at
  45 m/s are a Stage 4 check; the CAM blades' maximum rpm (printed on the pack) against the hover's ≈8 k rpm; the drive
  model beyond the pitch speed needs a bench check.
- **The FPV still wins** the hover (24 min with 1 kg against 13) and the mass (3.3 kg against 7.5).